In [1]:
import pandas as pd
from pathlib import Path

raw_path = Path("../data/raw")
cleaned_path = Path("../data/cleaned")

cleaned_path.mkdir(parents=True, exist_ok=True)

In [3]:
customers = pd.read_csv(raw_path / "customers.csv")
transactions = pd.read_csv(raw_path / "transactions.csv")
transaction_types = pd.read_csv(raw_path / "transaction_types.csv")
bill_payments = pd.read_csv(raw_path / "bill_payments.csv")
investments = pd.read_csv(raw_path / "investments.csv")
customer_support = pd.read_csv(raw_path / "customer_support.csv")

In [4]:
tables = {
    "customers": customers,
    "transactions": transactions,
    "transaction_types": transaction_types,
    "bill_payments": bill_payments,
    "investments": investments,
    "customer_support": customer_support
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

customers: (200, 5)
transactions: (1000, 5)
transaction_types: (6, 5)
bill_payments: (500, 5)
investments: (300, 5)
customer_support: (400, 5)


## Data Quality Audit

In [5]:
for name, df in tables.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print(f"{'=' * 60}")
    
    print("\nShape:", df.shape)
    
    print("\nData Types:")
    print(df.dtypes)
    
    print("\nMissing Values:")
    print(df.isnull().sum())
    
    print("\nDuplicate Rows:", df.duplicated().sum())


CUSTOMERS

Shape: (200, 5)

Data Types:
id            int64
full_name       str
email           str
phone           str
created_at      str
dtype: object

Missing Values:
id             0
full_name      0
email          0
phone         20
created_at     0
dtype: int64

Duplicate Rows: 0

TRANSACTIONS

Shape: (1000, 5)

Data Types:
id                    int64
customer_id           int64
amount              float64
transaction_type        str
transaction_date        str
dtype: object

Missing Values:
id                    0
customer_id           0
amount              103
transaction_type      0
transaction_date      0
dtype: int64

Duplicate Rows: 0

TRANSACTION_TYPES

Shape: (6, 5)

Data Types:
id                  int64
name                  str
description           str
category              str
average_amount    float64
dtype: object

Missing Values:
id                0
name              0
description       0
category          0
average_amount    0
dtype: int64

Duplicate Rows: 0

BI

##  check unique values

In [6]:
for name, df in tables.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print(f"{'=' * 60}")
    
    for column in df.select_dtypes(include="object").columns:
        print(f"\n{column}:")
        print(df[column].value_counts(dropna=False).head(10))


CUSTOMERS

full_name:
full_name
David Pereira    2
Mei Andersen     2
Amir Ibrahim     1
Yusuf Ali        1
Jack Patel       1
Nour Park        1
Nina Sousa       1
Wei Ali          1
Leo Meyer        1
Carlos Khan      1
Name: count, dtype: int64

email:
email
david.pereira@example.net    2
amir.ibrahim@example.net     1
yusuf.ali@example.org        1
jack.patel@example.net       1
nour.park@example.net        1
nina.sousa@example.com       1
wei.ali@example.net          1
leo.meyer@example.net        1
carlos.khan@example.com      1
maria.smith@example.org      1
Name: count, dtype: int64

phone:
phone
NaN                20
+1-469-555-0122     3
+1-612-555-0161     2
+1-212-555-0131     2
+1-213-555-0135     2
+1-415-555-0182     1
+1-214-555-0191     1
+1-512-555-0129     1
+1-213-555-0105     1
+1-972-555-0104     1
Name: count, dtype: int64

created_at:
created_at
2020-03-17 14:15:14    1
2020-08-16 14:13:08    1
2021-10-21 12:42:23    1
2023-12-03 10:14:40    1
2021-09-02 08:06:

C:\Users\star\AppData\Local\Temp\ipykernel_2220\1218946261.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df.select_dtypes(include="object").columns:
C:\Users\star\AppData\Local\Temp\ipykernel_2220\1218946261.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

In [7]:
quality_report = []

for name, df in tables.items():
    for column in df.columns:
        quality_report.append({
            "table": name,
            "column": column,
            "dtype": str(df[column].dtype),
            "rows": len(df),
            "missing": df[column].isna().sum(),
            "missing_pct": round(df[column].isna().mean() * 100, 2),
            "unique": df[column].nunique(dropna=True)
        })

quality_report = pd.DataFrame(quality_report)

quality_report

,table,column,dtype,rows,missing,missing_pct,unique
0,customers,id,int64,200,0,0.0,200
1,customers,full_name,str,200,0,0.0,198
2,customers,email,str,200,0,0.0,199
3,customers,phone,str,200,20,10.0,175
4,customers,created_at,str,200,0,0.0,200
5,transactions,id,int64,1000,0,0.0,1000
6,transactions,customer_id,int64,1000,0,0.0,198
7,transactions,amount,float64,1000,103,10.3,896
8,transactions,transaction_type,str,1000,0,0.0,2
9,transactions,transaction_date,str,1000,0,0.0,1000


In [8]:
print("Customer IDs:", customers["id"].nunique())

print("Transaction customer IDs:",
      transactions["customer_id"].nunique())

print("Bill payment customer IDs:",
      bill_payments["customer_id"].nunique())

print("Investment customer IDs:",
      investments["customer_id"].nunique())

print("Support customer IDs:",
      customer_support["customer_id"].nunique())

Customer IDs: 200
Transaction customer IDs: 198
Bill payment customer IDs: 182
Investment customer IDs: 155
Support customer IDs: 171


In [9]:
customer_ids = set(customers["id"])

for name, df in {
    "transactions": transactions,
    "bill_payments": bill_payments,
    "investments": investments,
    "customer_support": customer_support
}.items():
    
    invalid_ids = set(df["customer_id"].dropna()) - customer_ids
    
    print(f"{name}: {len(invalid_ids)} invalid customer IDs")

transactions: 0 invalid customer IDs
bill_payments: 0 invalid customer IDs
investments: 0 invalid customer IDs
customer_support: 0 invalid customer IDs


In [10]:
categorical_columns = {
    "transactions": ["transaction_type"],
    "transaction_types": ["name", "category"],
    "bill_payments": ["bill_type"],
    "investments": ["investment_type"],
    "customer_support": ["issue_type", "resolution_status"]
}

for table_name, columns in categorical_columns.items():
    print(f"\n{'=' * 50}")
    print(table_name.upper())
    
    for column in columns:
        print(f"\n{column}:")
        print(tables[table_name][column].value_counts(dropna=False))


TRANSACTIONS

transaction_type:
transaction_type
deposit       677
withdrawal    323
Name: count, dtype: int64

TRANSACTION_TYPES

name:
name
deposit statement             1
loan exchange                 1
fee adjustment                1
purchase                      1
credit                        1
exchange credit adjustment    1
Name: count, dtype: int64

category:
category
personal    3
business    3
Name: count, dtype: int64

BILL_PAYMENTS

bill_type:
bill_type
utility        272
credit_card    182
mortgage        46
Name: count, dtype: int64

INVESTMENTS

investment_type:
investment_type
stocks          188
bonds            78
mutual_funds     34
Name: count, dtype: int64

CUSTOMER_SUPPORT

issue_type:
issue_type
account        288
transaction     77
technical       35
Name: count, dtype: int64

resolution_status:
resolution_status
resolved    364
pending      36
Name: count, dtype: int64


## Step 2 — Data Cleaning

In [11]:
date_columns = {
    "customers": ["created_at"],
    "transactions": ["transaction_date"],
    "bill_payments": ["payment_date"],
    "investments": ["investment_date"],
    "customer_support": ["interaction_date"]
}

for table_name, columns in date_columns.items():
    for column in columns:
        tables[table_name][column] = pd.to_datetime(
            tables[table_name][column],
            errors="coerce"
        )

In [12]:
for table_name, columns in date_columns.items():
    for column in columns:
        print(
            f"{table_name}.{column} → "
            f"dtype: {tables[table_name][column].dtype}, "
            f"invalid dates: {tables[table_name][column].isna().sum()}"
        )

customers.created_at → dtype: datetime64[us], invalid dates: 0
transactions.transaction_date → dtype: datetime64[us], invalid dates: 0
bill_payments.payment_date → dtype: datetime64[us], invalid dates: 0
investments.investment_date → dtype: datetime64[us], invalid dates: 0
customer_support.interaction_date → dtype: datetime64[us], invalid dates: 0


In [13]:
for name, df in tables.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

customers: 0 duplicate rows
transactions: 0 duplicate rows
transaction_types: 0 duplicate rows
bill_payments: 0 duplicate rows
investments: 0 duplicate rows
customer_support: 0 duplicate rows


In [14]:
for name, df in tables.items():
    df.to_csv(
        cleaned_path / f"{name}_cleaned.csv",
        index=False
    )

print("All cleaned files saved successfully.")

All cleaned files saved successfully.


In [15]:
for name, df in tables.items():
    print(
        f"{name}: "
        f"rows={len(df)}, "
        f"columns={len(df.columns)}, "
        f"duplicates={df.duplicated().sum()}"
    )

customers: rows=200, columns=5, duplicates=0
transactions: rows=1000, columns=5, duplicates=0
transaction_types: rows=6, columns=5, duplicates=0
bill_payments: rows=500, columns=5, duplicates=0
investments: rows=300, columns=5, duplicates=0
customer_support: rows=400, columns=5, duplicates=0


## Data Cleaning Summary

- Converted all date columns from string format to datetime format.
- Checked all tables for duplicate rows; no duplicates were found.
- Validated customer_id relationships across transaction, bill payment, investment, and customer support tables; no invalid customer IDs were found.
- Preserved missing financial amounts as NULL instead of applying statistical imputation to avoid introducing artificial monetary values.
- Preserved missing phone numbers as NULL.
- Saved the cleaned datasets separately from the raw datasets.